In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.fid import FrechetInceptionDistance

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
latent_dim = 100
num_epochs = 50
batch_size = 128
learning_rate = 0.0002
image_size = 32
channels = 3

# Create a directory to save generated images
os.makedirs("generated_images", exist_ok=True)

# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Define the Generator class
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, channels * image_size * image_size),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), channels, image_size, image_size)
        return img

# Define the Discriminator class
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(channels * image_size * image_size, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

# Initialize models
generator = Generator().to(device)
discriminator = Discriminator().to(device)

# Loss function and optimizers
criterion = nn.BCELoss()
g_optimizer = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

# Training loop
for epoch in range(1, num_epochs + 1):
    for i, (real_images, _) in enumerate(dataloader):
        batch_size = real_images.size(0)
        real_images = real_images.to(device)

        # Generate labels
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Train Discriminator
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(z)

        real_loss = criterion(discriminator(real_images), real_labels)
        fake_loss = criterion(discriminator(fake_images.detach()), fake_labels)
        d_loss = real_loss + fake_loss

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # Train Generator
        g_loss = criterion(discriminator(fake_images), real_labels)
        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    # Save generated images
    epoch_dir = f"generated_images/Epoch {epoch}"
    os.makedirs(epoch_dir, exist_ok=True)
    save_image(fake_images[:25], os.path.join(epoch_dir, "generated.png"), nrow=5, normalize=True)
    print(f"Epoch [{epoch}/{num_epochs}] | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

# Evaluation
inception = InceptionScore().to(device)
fid = FrechetInceptionDistance(feature=2048).to(device)

generated_images = []
real_images = []

for _ in range(10):  # Generate multiple batches for evaluation
    z = torch.randn(batch_size, latent_dim).to(device)
    gen_imgs = generator(z).detach().cpu()
    generated_images.append(gen_imgs)

generated_images = torch.cat(generated_images)
real_images = torch.stack([dataset[i][0] for i in range(len(generated_images))])

# Compute Inception Score
is_mean, is_std = inception(generated_images)
print(f"Inception Score: {is_mean:.4f} ± {is_std:.4f}")

# Compute FID
fid.update(real_images, real=True)
fid.update(generated_images, real=False)
fid_score = fid.compute()
print(f"FID Score: {fid_score:.4f}")

100%|██████████| 170M/170M [00:02<00:00, 71.1MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Epoch [1/50] | D Loss: 0.2642 | G Loss: 6.3121
Epoch [2/50] | D Loss: 0.4944 | G Loss: 4.4759
Epoch [3/50] | D Loss: 0.1234 | G Loss: 4.7175
Epoch [4/50] | D Loss: 0.6074 | G Loss: 6.6003
Epoch [5/50] | D Loss: 0.1678 | G Loss: 4.5439
Epoch [6/50] | D Loss: 0.1119 | G Loss: 4.8584
Epoch [7/50] | D Loss: 0.0810 | G Loss: 4.4878
Epoch [8/50] | D Loss: 0.1110 | G Loss: 4.0367
Epoch [9/50] | D Loss: 0.3366 | G Loss: 4.0937
Epoch [10/50] | D Loss: 0.2219 | G Loss: 4.7708
Epoch [11/50] | D Loss: 0.4089 | G Loss: 3.1709
Epoch [12/50] | D Loss: 0.4568 | G Loss: 3.2887
Epoch [13/50] | D Loss: 0.7888 | G Loss: 2.6737
Epoch [14/50] | D Loss: 0.7102 | G Loss: 2.4129
Epoch [15/50] | D Loss: 0.5007 | G Loss: 2.8502
Epoch [16/50] | D Loss: 0.6222 | G Loss: 2.5028
Epoch [17/50] | D Loss: 0.8608 | G Loss: 2.3107
Epoch [18/50] | D Loss: 0.8208 | G Loss: 2.4242
Epoch [19/50] | D Loss: 0.8290 | G Loss: 2.3207
Epoch [20/50] | D Loss: 0.7554 | G Loss: 2.431

/usr/local/lib/python3.11/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


ModuleNotFoundError: InceptionScore metric requires that `Torch-fidelity` is installed. Either install as `pip install torchmetrics[image]` or `pip install torch-fidelity`.

In [4]:
# Function to generate new images
def generate_new_images(num_images=25, output_dir="generated_new"):
    os.makedirs(output_dir, exist_ok=True)
    z = torch.randn(num_images, latent_dim).to(device)
    generated_images = generator(z).detach()
    save_image(generated_images, os.path.join(output_dir, "new_generated.png"), nrow=5, normalize=True)
    print(f"Generated {num_images} new images saved in {output_dir}")

# Example usage
generate_new_images()

Generated 25 new images saved in generated_new


In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.6/931.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli